# Lecture 24 — Machine Learning Compilation in Code

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

The lecture uses Apache TVM: write a loop program, transform it with calls like `split` and `reorder`, build
it, measure, and search for the fastest variant. TVM is not installed here, so this notebook builds a small
version of the same idea from scratch:

1. a loop nest held as **data**, turned into real Python code
2. `split` and `reorder` as functions that rewrite that data — every variant stays correct
3. a cost to measure: cache misses in a simulated cache
4. automatic search over schedules — **tuning**
5. fusing a bias add into the loops

In [1]:
import numpy as np, random, itertools
from collections import OrderedDict

## 1. A loop program as data

A matrix multiply $C_{ij} = \sum_k A_{ik}B_{kj}$ is a list of loops plus a body. `build` turns that description
into Python source and compiles it — the same role as `tvm.build`.

In [2]:
class Schedule:
    def __init__(self, n):
        self.loops = [("i", n), ("j", n), ("k", n)]           # (variable, extent), outer to inner
        self.expr = {"i": "i", "j": "j", "k": "k"}            # how each original index is computed

    def split(self, var, factor):
        """One loop of extent n becomes an outer loop (n // factor) and an inner loop (factor)."""
        idx = [v for v, _ in self.loops].index(var)
        n = self.loops[idx][1]
        outer, inner = var + "o", var + "i"
        self.loops[idx:idx + 1] = [(outer, n // factor), (inner, factor)]
        for k, e in self.expr.items():
            self.expr[k] = e.replace(var, f"({outer}*{factor}+{inner})") if e == var or var in e.split() else e
        if self.expr.get(var[0], "") == var:
            self.expr[var[0]] = f"({outer}*{factor}+{inner})"
        return outer, inner

    def reorder(self, order):
        ext = dict(self.loops)
        self.loops = [(v, ext[v]) for v in order]

    def source(self, instrument=False):
        lines = ["def kernel(A, B, C, access):"]
        for depth, (v, ext) in enumerate(self.loops, start=1):
            lines.append("    " * depth + f"for {v} in range({ext}):")
        i, j, k = self.expr["i"], self.expr["j"], self.expr["k"]
        pad = "    " * (len(self.loops) + 1)
        if instrument:
            lines.append(pad + f"access(0, {i}, {k}); access(1, {k}, {j})")
        lines.append(pad + f"C[{i}, {j}] += A[{i}, {k}] * B[{k}, {j}]")
        return "\n".join(lines)

    def build(self, instrument=False):
        scope = {}
        exec(self.source(instrument), scope)
        return scope["kernel"]

s = Schedule(32)
print(s.source())

def kernel(A, B, C, access):
    for i in range(32):
        for j in range(32):
            for k in range(32):
                C[i, j] += A[i, k] * B[k, j]


In [3]:
n = 32
A, B = np.random.rand(n, n), np.random.rand(n, n)
C = np.zeros((n, n))
Schedule(n).build()(A, B, C, None)
print("built kernel computes A @ B:", np.allclose(C, A @ B))

built kernel computes A @ B: True


## 2. Transforming the program

`split` and `reorder` rewrite the description; the code is regenerated. This is the lecture's point: change the
loop structure with a function call instead of hand-editing nested loops.

In [4]:
s = Schedule(32)
io, ii = s.split("i", 8)
jo, ji = s.split("j", 8)
s.reorder([io, jo, "k", ii, ji])                             # a blocked loop order
print(s.source())

C = np.zeros((n, n))
s.build()(A, B, C, None)
print("\ntransformed kernel still computes A @ B:", np.allclose(C, A @ B))

def kernel(A, B, C, access):
    for io in range(4):
        for jo in range(4):
            for k in range(32):
                for ii in range(8):
                    for ji in range(8):
                        C[(io*8+ii), (jo*8+ji)] += A[(io*8+ii), k] * B[k, (jo*8+ji)]

transformed kernel still computes A @ B: True


## 3. Measuring a schedule

TVM times each variant on real hardware. Python loops would drown that signal, so instead feed every memory
read into a simulated cache — 64 lines of 8 numbers — and count misses. Same arithmetic, different order,
different cost.

In [5]:
def misses(schedule, n, lines=64, line_elems=8):
    cache = OrderedDict()
    count = [0]
    def access(which, r, c):
        line = (which, (r * n + c) // line_elems)
        if line in cache:
            cache.move_to_end(line)
        else:
            count[0] += 1
            cache[line] = True
            if len(cache) > lines:
                cache.popitem(last=False)
    A_, B_, C_ = np.zeros((n, n)), np.zeros((n, n)), np.zeros((n, n))
    schedule.build(instrument=True)(A_, B_, C_, access)
    return count[0]

n = 48
base = Schedule(n)
s2 = Schedule(n); io, ii = s2.split("i", 8); jo, ji = s2.split("j", 8)
s2.reorder([io, jo, "k", ii, ji])                           # the blocked schedule from section 2
s3 = Schedule(n); s3.reorder(["i", "k", "j"])
print("default i, j, k           :", f"{misses(base, n):>7,} misses")
print("only reordered to i, k, j :", f"{misses(s3, n):>7,} misses")
print("split into 8x8 blocks     :", f"{misses(s2, n):>7,} misses")

default i, j, k           :  14,112 misses


only reordered to i, k, j :  14,112 misses


split into 8x8 blocks     :   3,456 misses


Reordering alone changes nothing here: in both orders every row of $C$ streams the whole of $B$ through a
cache too small to keep it. Blocking does change it — an $8\times8$ block of $C$ reuses the same strips of $A$
and $B$ many times while they are still cached. Which transformation helps depends on the sizes involved,
which is exactly why the next step is to search.

## 4. Tuning: search for the best schedule

There are many possible splits and orders, and which one wins depends on the machine. So the lecture's
approach is to generate many variants, measure each, and keep the fastest. A small random search:

In [6]:
def random_schedule(n, rng):
    s = Schedule(n)
    fi, fj, fk = (rng.choice([f for f in (2, 4, 8, 16) if n % f == 0]) for _ in range(3))
    io, ii = s.split("i", fi); jo, ji = s.split("j", fj); ko, ki = s.split("k", fk)
    outer = [io, jo, ko]; rng.shuffle(outer)
    inner = [ii, ji, ki]; rng.shuffle(inner)
    s.reorder(outer + inner)
    return s, (fi, fj, fk, outer + inner)

rng = random.Random(0)
trials = []
for _ in range(25):
    s, desc = random_schedule(n, rng)
    trials.append((misses(s, n), desc, s))
trials.sort(key=lambda t: t[0])

best_misses, best_desc, best = trials[0]
print(f"default schedule: {misses(Schedule(n), n):>7,} misses")
print(f"worst of 25     : {trials[-1][0]:>7,} misses")
print(f"best of 25      : {best_misses:>7,} misses   tiles {best_desc[:3]}, order {best_desc[3]}")
C = np.zeros((n, n)); Ab, Bb = np.random.rand(n, n), np.random.rand(n, n)
best.build()(Ab, Bb, C, None)
print("best schedule is still correct:", np.allclose(C, Ab @ Bb))

default schedule:  14,112 misses
worst of 25     :  10,368 misses
best of 25      :     576 misses   tiles (4, 4, 4), order ['ko', 'jo', 'io', 'ji', 'ki', 'ii']
best schedule is still correct: True


The search never needed to understand the hardware: it tried candidates, measured, and kept the winner. Real
tuners add a learned cost model to choose which candidates to try. In the lecture, TVM's matrix multiply went
from about 3 seconds to 0.37 just by splitting and reordering loops.

## 5. Fusion at the loop level

The lecture's example: a matrix multiply followed by a bias add is two passes over $C$. Initialise $C$ with the
bias instead of zero, and the second pass disappears.

In [7]:
n = 32
A, B, bias = np.random.rand(n, n), np.random.rand(n, n), np.random.rand(n)

def unfused(A, B, bias):
    C = np.zeros((n, n)); passes = 0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    for i in range(n):                                          # a second pass over C
        for j in range(n):
            C[i, j] += bias[j]; passes += 1
    return C, passes

def fused(A, B, bias):
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            C[i, j] = bias[j]                                   # start from the bias
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C, 0

C1, extra = unfused(A, B, bias)
C2, _ = fused(A, B, bias)
print("same result:", np.allclose(C1, C2), "| extra writes to C saved:", extra)

same result: True | extra writes to C saved: 1024


## Summary

- A loop program held as data can be rewritten by `split` and `reorder`, and rebuilt into working code — every
  variant here computed the same matrix product.
- Reordering alone did not help at this size; splitting into blocks did.
- A random search over 25 schedules found one far better than the default, with no hardware model at all.
- Fusing the bias into the matrix multiply removed a whole pass over the output.